# 🤖 Pipeline de 3 Agentes ML — Dataset Titanic

```
Dataset CSV (Titanic)
     ↓
AGENTE 1 - Normalizador   → limpia, imputa, escala, codifica
     ↓ dataset limpio
AGENTE 2 - Entrenador     → aplica validación, entrena, selecciona modelo
     ↓ métricas + modelo
AGENTE 3 - Comunicador    → genera reporte en lenguaje natural (Gemini)
```

**Objetivo:** Predecir si un pasajero del Titanic **sobrevivió** (`1`) o **no** (`0`).

## 📦 Paso 1 — Instalar dependencias

In [ ]:
!pip install pandas scikit-learn google-generativeai --quiet
print('✅ Dependencias instaladas')

## 🔑 Paso 2 — Tu API Key de Google (Gemini)

1. Entrá a **aistudio.google.com**
2. Hacé click en **Get API Key → Create API Key**
3. Copiá la clave y pegala abajo (empieza con `AIza...`)

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = 'AIza-tu-clave-aqui'
print('✅ API Key configurada')

## 🔧 Importaciones

In [ ]:
import pandas as pd
import numpy as np
import json
import google.generativeai as genai

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

print('✅ Importaciones exitosas')

## 📊 Paso 3 — Cargar el dataset Titanic

In [ ]:
# Se descarga automáticamente, no necesitás bajar ningún archivo
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df_raw = pd.read_csv(url)

# Columnas que vamos a usar:
# Pclass   = clase del ticket (1=primera, 2=segunda, 3=tercera)
# Sex      = sexo del pasajero
# Age      = edad
# SibSp    = hermanos/cónyuge a bordo
# Parch    = padres/hijos a bordo
# Fare     = precio del ticket
# Survived = 1 sobrevivió, 0 no  ← COLUMNA OBJETIVO

df_raw = df_raw[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Survived']]

print('📋 Dataset Titanic cargado:')
print(f'  Filas: {len(df_raw)} | Columnas: {len(df_raw.columns)}')
print(f'  Valores faltantes: {df_raw.isnull().sum().sum()}')
display(df_raw.head(10))

---
## 🤖 AGENTE 1 — Normalizador
> Limpia, imputa, escala y codifica el dataset

In [ ]:
class AgenteNormalizador:
    def __init__(self, target_col):
        self.target_col = target_col
        self.scaler = StandardScaler()
        self.imputer_num = SimpleImputer(strategy='median')
        self.imputer_cat = SimpleImputer(strategy='most_frequent')
        self.label_encoders = {}

    def ejecutar(self, df):
        print('🔄 [AGENTE 1] Iniciando normalización...')
        df = df.copy()

        y = df[self.target_col]
        X = df.drop(columns=[self.target_col])

        cols_num = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
        cols_cat = X.select_dtypes(include=['object']).columns.tolist()

        if cols_num:
            X[cols_num] = self.imputer_num.fit_transform(X[cols_num])
            print(f'  ✔ Imputadas columnas numéricas: {cols_num}')

        if cols_cat:
            X[cols_cat] = self.imputer_cat.fit_transform(X[cols_cat])
            for col in cols_cat:
                le = LabelEncoder()
                X[col] = le.fit_transform(X[col])
                self.label_encoders[col] = le
            print(f'  ✔ Codificadas columnas categóricas: {cols_cat}')

        if cols_num:
            X[cols_num] = self.scaler.fit_transform(X[cols_num])
            print(f'  ✔ Escaladas columnas numéricas')

        df_limpio = X.copy()
        df_limpio[self.target_col] = y.values

        print(f'  ✅ Dataset limpio listo: {df_limpio.shape[0]} filas × {df_limpio.shape[1]} columnas')
        return df_limpio


TARGET = 'Survived'

agente1 = AgenteNormalizador(target_col=TARGET)
df_limpio = agente1.ejecutar(df_raw)

print('\n📋 Dataset limpio:')
display(df_limpio.head())

---
## 🤖 AGENTE 2 — Entrenador
> Compara modelos con validación cruzada y selecciona el mejor

In [ ]:
class AgenteEntrenador:
    def __init__(self, target_col, cv_folds=5):
        self.target_col = target_col
        self.cv_folds = cv_folds
        self.modelos_candidatos = {
            'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
            'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting':   GradientBoostingClassifier(random_state=42),
            'SVM':                 SVC(kernel='rbf', random_state=42)
        }
        self.mejor_modelo = None
        self.mejor_nombre = None
        self.resultados = {}

    def ejecutar(self, df):
        print('🔄 [AGENTE 2] Iniciando entrenamiento...')

        X = df.drop(columns=[self.target_col]).values
        y = df[self.target_col].values

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        print(f'  Train: {len(X_train)} | Test: {len(X_test)} | CV: {self.cv_folds} folds\n')

        mejor_score = -np.inf

        for nombre, modelo in self.modelos_candidatos.items():
            scores = cross_val_score(modelo, X_train, y_train, cv=self.cv_folds, scoring='accuracy')
            self.resultados[nombre] = {'cv_mean': round(scores.mean(), 4), 'cv_std': round(scores.std(), 4)}
            print(f'  📊 {nombre:<25} Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')
            if scores.mean() > mejor_score:
                mejor_score = scores.mean()
                self.mejor_nombre = nombre
                self.mejor_modelo = modelo

        self.mejor_modelo.fit(X_train, y_train)
        y_pred = self.mejor_modelo.predict(X_test)
        reporte = classification_report(y_test, y_pred, output_dict=True)

        metricas = {
            'modelo_seleccionado': self.mejor_nombre,
            'accuracy_test': round(accuracy_score(y_test, y_pred), 4),
            'cv_accuracy_mean': self.resultados[self.mejor_nombre]['cv_mean'],
            'cv_accuracy_std': self.resultados[self.mejor_nombre]['cv_std'],
            'precision_macro': round(reporte['macro avg']['precision'], 4),
            'recall_macro': round(reporte['macro avg']['recall'], 4),
            'f1_macro': round(reporte['macro avg']['f1-score'], 4),
            'comparacion_modelos': self.resultados
        }

        print(f'\n  🏆 Mejor modelo: {self.mejor_nombre}')
        print(f'  ✅ Accuracy en test: {metricas["accuracy_test"]}')
        return self.mejor_modelo, metricas


agente2 = AgenteEntrenador(target_col=TARGET)
modelo_final, metricas = agente2.ejecutar(df_limpio)

print('\n📈 Métricas:')
print(json.dumps({k: v for k, v in metricas.items() if k != 'comparacion_modelos'}, indent=2))

---
## 🤖 AGENTE 3 — Comunicador (Gemini)
> Genera el reporte final en lenguaje natural usando Google Gemini

In [ ]:
class AgenteComunicador:
    def __init__(self):
        genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
        self.modelo = genai.GenerativeModel('gemini-1.5-flash')

    def ejecutar(self, metricas, contexto=''):
        print('🔄 [AGENTE 3] Generando reporte con Gemini...\n')

        prompt = f"""Eres un analista de datos experto. Analizá los resultados de este pipeline 
de machine learning y generá un reporte ejecutivo claro en español.

Contexto: {contexto}

Métricas:
{json.dumps(metricas, indent=2, ensure_ascii=False)}

El reporte debe incluir:
1. Resumen ejecutivo (2-3 oraciones)
2. Modelo seleccionado y por qué ganó
3. Qué significan los números (accuracy, precision, recall, F1) en términos simples
4. Comparación breve con los otros modelos
5. Conclusión y recomendación final

Usá lenguaje accesible para alguien sin experiencia técnica."""

        respuesta = self.modelo.generate_content(prompt)
        return respuesta.text


agente3 = AgenteComunicador()
reporte = agente3.ejecutar(
    metricas=metricas,
    contexto='Predicción de supervivencia de pasajeros del Titanic'
)

print('=' * 70)
print('📄 REPORTE EJECUTIVO')
print('=' * 70)
print(reporte)
print('=' * 70)